# Exercise 2: Context Engineering Lab

In this exercise you will practice techniques for managing what goes into the context window:

1. Measuring prompt size & building a context budget
2. Compressing long conversation histories
3. Dynamic tool selection from a larger registry
4. Context positioning experiments
5. Simple trace logging for observability

**Prerequisites**: Lecture 7 slides, basic LangChain / LangGraph familiarity.

## Setup

In [1]:
!pip install langchain langchain-openai tiktoken --quiet


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that the required environment variables are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [5]:
import tiktoken
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Initialize the LLM
#llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")
llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="model-group3")
enc = tiktoken.encoding_for_model("gpt-5")
#enc = tiktoken.encoding_for_model("model-group3")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def messages_tokens(messages) -> int:
    return sum(count_tokens(m.content) for m in messages)

## Part 1: Context Budget Tracker

We budget the window across sections. The tracker tells us how close we are to our cap.

In [6]:
class ContextBudget:
    def __init__(self, total=20_000, reserved_reply=1_500):
        self.total = total
        self.reserved_reply = reserved_reply
        self.sections = {}

    def add(self, name: str, text: str):
        self.sections[name] = count_tokens(text)

    def used(self) -> int:
        return sum(self.sections.values())

    def available(self) -> int:
        return self.total - self.reserved_reply - self.used()

    def report(self):
        print(f"Budget: {self.total} tokens (reply reserved: {self.reserved_reply})")
        for k, v in self.sections.items():
            print(f"  {k:<25} {v:>6} tokens")
        print(f"  {'USED':<25} {self.used():>6}")
        print(f"  {'AVAILABLE':<25} {self.available():>6}")

budget = ContextBudget(total=16_000)
budget.add("system_prompt", "You are a helpful assistant. " * 50)
budget.add("tool_defs", "search(query) -> str\n" * 20)
budget.add("retrieved_docs", "lorem ipsum " * 800)
budget.add("chat_history", "Hello! How are you? " * 200)
budget.add("user_query", "Summarize the report about Q4 revenue.")
budget.report()

Budget: 16000 tokens (reply reserved: 1500)
  system_prompt                301 tokens
  tool_defs                    120 tokens
  retrieved_docs              1602 tokens
  chat_history                1201 tokens
  user_query                    10 tokens
  USED                        3234
  AVAILABLE                  11266


**TODO**: If `available()` goes negative, what do you drop or compress first? Write a policy function:

In [7]:
def enforce_budget(budget: ContextBudget, sections: dict[str, str]) -> dict[str, str]:
    """Return possibly-shortened sections so the budget fits.
    Priority order (drop / truncate LAST = most important):
      system_prompt > user_query > tool_defs > retrieved_docs > chat_history
    """
    # TODO: implement truncation/dropping based on priority
    return sections

# Test it on the over-budget example above.

## Part 2: History Compression

Long chat histories blow the budget. Keep the recent messages verbatim and summarize the older ones.

In [8]:
history = [
    HumanMessage(content="Hi! I'm Alice, a data scientist."),
    AIMessage(content="Hi Alice! How can I help?"),
    HumanMessage(content="I'm working on a churn-prediction model for a telco."),
    AIMessage(content="Nice. What features do you have?"),
    HumanMessage(content="Monthly usage, tenure, contract type, complaints."),
    AIMessage(content="Good. Class imbalance is common here - try SMOTE or class weights."),
    HumanMessage(content="We picked XGBoost and got 0.82 AUC."),
    AIMessage(content="Solid baseline. Have you tried calibration?"),
    HumanMessage(content="Not yet. Deadline is next Friday. My teammate Bob handles deployment."),
    AIMessage(content="Noted - deadline Friday, Bob on deploy."),
    HumanMessage(content="What should I do next?"),
]

print(f"Total history tokens: {messages_tokens(history)}")

Total history tokens: 116


In [9]:
def compress_history(messages, keep_last: int = 4) -> list:
    if len(messages) <= keep_last:
        return messages
    old = messages[:-keep_last]
    recent = messages[-keep_last:]
    old_text = "\n".join(f"{m.type}: {m.content}" for m in old)
    summary_msg = llm.invoke(
        f"Summarize the following conversation in under 80 words, "
        f"preserving names, decisions, and open questions:\n\n{old_text}"
    ).content
    return [SystemMessage(content=f"Earlier conversation summary: {summary_msg}")] + recent

compressed = compress_history(history, keep_last=3)
for m in compressed:
    print(f"[{m.type}] {m.content}\n")

print(f"Original tokens:   {messages_tokens(history)}")
print(f"Compressed tokens: {messages_tokens(compressed)}")

[system] Earlier conversation summary: Alice, a data scientist, is building a telco churn-prediction model using monthly usage, tenure, contract type, and complaints. XGBoost was chosen and achieved 0.82 AUC. The assistant suggested handling class imbalance with SMOTE or class weights. Open question: whether calibration has been tried for the model.

[human] Not yet. Deadline is next Friday. My teammate Bob handles deployment.

[ai] Noted - deadline Friday, Bob on deploy.

[human] What should I do next?

Original tokens:   116
Compressed tokens: 101


**TODO**:

1. Ask the compressed conversation a question that depends on the old info (e.g. "What is the deadline?"). Does it still know?
2. Try `keep_last = 1` and see what breaks.
3. Improve the summary prompt to explicitly preserve named entities and dates.

## Part 3: Dynamic Tool Selection

Exposing 30 tools to an agent hurts accuracy and blows context. Instead, pick only the most relevant ones per query.

We'll fake it with simple keyword retrieval; in a real system you'd embed tool descriptions.

In [10]:
TOOL_REGISTRY = [
    {"name": "web_search", "desc": "Search the public internet for current information"},
    {"name": "calculator", "desc": "Evaluate arithmetic and math expressions"},
    {"name": "calendar_lookup", "desc": "Look up events on the user's calendar"},
    {"name": "calendar_create", "desc": "Create a new event on the user's calendar"},
    {"name": "email_send", "desc": "Send an email on behalf of the user"},
    {"name": "email_search", "desc": "Search the user's email inbox"},
    {"name": "jira_create", "desc": "Create a Jira ticket"},
    {"name": "jira_search", "desc": "Search Jira tickets"},
    {"name": "code_execute", "desc": "Run Python code in a sandbox"},
    {"name": "weather", "desc": "Get current weather for a city"},
    {"name": "stock_price", "desc": "Get the current stock price of a ticker"},
    {"name": "translate", "desc": "Translate text between languages"},
]

def select_tools(query: str, k: int = 3) -> list[dict]:
    q_words = set(query.lower().split())
    scored = []
    for tool in TOOL_REGISTRY:
        doc_words = set((tool["name"] + " " + tool["desc"]).lower().split())
        score = len(q_words & doc_words)
        scored.append((score, tool))
    scored.sort(key=lambda x: -x[0])
    return [t for s, t in scored[:k]]

for q in [
    "What's the weather in Berlin?",
    "Schedule a meeting with Bob tomorrow at 3pm",
    "Translate this email to Spanish and send it",
    "Compute 17 * 233 + 42",
]:
    picked = [t["name"] for t in select_tools(q)]
    print(f"{q}\n  -> {picked}\n")

What's the weather in Berlin?
  -> ['web_search', 'calendar_lookup', 'calendar_create']

Schedule a meeting with Bob tomorrow at 3pm
  -> ['calendar_create', 'jira_create', 'code_execute']

Translate this email to Spanish and send it
  -> ['email_send', 'calculator', 'email_search']

Compute 17 * 233 + 42
  -> ['web_search', 'calculator', 'calendar_lookup']



**TODO**: Swap the keyword scorer for a real embedding-based retriever (use `OpenAIEmbeddings` + cosine similarity). Compare which queries benefit most.

## Part 4: Context Position Experiment

Does putting the critical instruction at the START vs END of the context change the answer?

In [11]:
# We hide a specific constraint among a lot of filler and see if the model obeys it.
filler = ("You may also consider historical trends and seasonal effects. " * 30)

critical_rule = "The answer MUST always be returned in UPPERCASE."

prompt_start = (
    f"{critical_rule}\n\n{filler}\n\n"
    f"Question: What is the capital of France?"
)

prompt_middle = (
    f"{filler[:len(filler)//2]}\n\n"
    f"{critical_rule}\n\n"
    f"{filler[len(filler)//2:]}\n\n"
    f"Question: What is the capital of France?"
)

prompt_end = (
    f"{filler}\n\n"
    f"{critical_rule}\n\n"
    f"Question: What is the capital of France?"
)

for name, p in [("START", prompt_start), ("MIDDLE", prompt_middle), ("END", prompt_end)]:
    out = llm.invoke(p).content
    obeyed = out == out.upper()
    print(f"{name:<7} obeyed_uppercase={obeyed} out={out!r}")

START   obeyed_uppercase=True out='PARIS'
MIDDLE  obeyed_uppercase=True out='PARIS'
END     obeyed_uppercase=True out='PARIS'


**TODO**: Run each variant 5 times (temperature=1.0). Which position has the highest obedience rate?

## Part 5: Simple Trace Logger

A minimal tracer so you can inspect what was actually sent to the model.

In [12]:
import json, time

TRACE_LOG = []

def traced_invoke(messages, meta=None):
    t0 = time.time()
    tokens_in = messages_tokens(messages)
    reply = llm.invoke(messages)
    dt = time.time() - t0
    entry = {
        "ts": t0,
        "meta": meta or {},
        "tokens_in": tokens_in,
        "tokens_out": count_tokens(reply.content),
        "latency_s": round(dt, 2),
        "preview": reply.content[:80],
    }
    TRACE_LOG.append(entry)
    return reply

traced_invoke(
    [SystemMessage(content="You are a poet."),
     HumanMessage(content="Write a 2-line poem about context windows.")],
    meta={"experiment": "poetry_v1"}
)

print(json.dumps(TRACE_LOG, indent=2))

[
  {
    "ts": 1777482272.401058,
    "meta": {
      "experiment": "poetry_v1"
    },
    "tokens_in": 15,
    "tokens_out": 22,
    "latency_s": 0.86,
    "preview": "In the window of context, the past leans near,  \nA brief little sky where meanin"
  }
]


**TODO (challenge)**:

1. Combine everything: build an agent call that uses `select_tools`, `compress_history`, and `enforce_budget`.
2. Log each call with `traced_invoke` + metadata.
3. Replay a 20-turn conversation and chart `tokens_in` over turns. When does compression kick in?

## Reflection

- Which piece of context gave you the biggest saving for the smallest quality loss?
- How did tool count affect answer quality / speed?
- If you could only keep 1 observability metric, which would it be and why?